In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import os
import tensorflow as tf

2026-05-06 14:25:18.199306: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778077518.395825      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778077518.450287      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778077518.880113      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778077518.880158      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778077518.880161      57 computation_placer.cc:177] computation placer alr

In [18]:
!cp "/kaggle/input/datasets/adelhefny/arsl-landmarks-npy-part2/test/0001/01_01_0001_(10_11_16_16_20_30)_c.npy" /kaggle/working/

In [19]:
import os
import numpy as np
from sklearn.model_selection import train_test_split

base_dirs = [
    "/kaggle/input/datasets/adelhefny/arsl-landmarks-npy-part2",
    "/kaggle/input/datasets/adelhefny/arsl-landmarks-npy-part1",
    "/kaggle/input/datasets/adelhefny/arsl-landmarks-npy-part3"
]

def get_paths_and_labels(base_dirs, data_type):
    paths, labels = [], []
    unique_labels = sorted(os.listdir(os.path.join(base_dirs[0], "train")))
    label_to_index = {name: i for i, name in enumerate(unique_labels)}

    for base_dir in base_dirs:
        data_path = os.path.join(base_dir, data_type)
        if not os.path.exists(data_path): continue

        for label in unique_labels:
            label_dir = os.path.join(data_path, label)
            if os.path.exists(label_dir):
                for file in os.listdir(label_dir):
                    paths.append(os.path.join(label_dir, file))
                    labels.append(label_to_index[label])
    return paths, labels, len(unique_labels)

train_val_paths, train_val_labels, num_classes = get_paths_and_labels(base_dirs, "train")
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths,
    train_val_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_val_labels
)

test_paths, test_labels, _ = get_paths_and_labels(base_dirs, "test")

In [8]:
import numpy as np
from tqdm import tqdm

def calculate_average_frames(paths):
    frame_counts = []
    print(f'Processing {len(paths)} files...')
    for path in tqdm(paths):
        data = np.load(path, mmap_mode='r')
        frame_counts.append(data.shape[0])
    
    avg_frames = np.mean(frame_counts)
    max_frames = np.max(frame_counts)
    min_frames = np.min(frame_counts)
    
    return avg_frames, min_frames, max_frames

# avg_f, min_f, max_f = calculate_average_frames(train_paths)

# print(f'\nResults for Training Set:')
# print(f'Average frames: {avg_f:.2f}')
# print(f'Min frames:     {min_f}')
# print(f'Max frames:     {max_f}')

Processing 50774 files...


100%|██████████| 50774/50774 [02:15<00:00, 374.62it/s]


Results for Training Set:
Average frames: 20.71
Min frames:     3
Max frames:     135


### Efficient Data Pre-processing: Converting to TFRecords
To solve the GPU bottleneck, we will convert the `.npy` files into `TFRecord` files. This allows TensorFlow to stream data directly from disk to the GPU without being blocked by the Python Global Interpreter Lock (GIL) or NumPy overhead.

In [20]:
import tensorflow as tf
import numpy as np
import tqdm

def _float_feature(value):
    return tf.train.Feature(float_list=tf.train.FloatList(value=value))

def _int64_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

def create_tfrecord(paths, labels, output_file):
    with tf.io.TFRecordWriter(output_file) as writer:
        for path, label in tqdm.tqdm(zip(paths, labels), total=len(paths)):
            data = np.load(path).astype(np.float32)
            flattened_data = data.flatten().tolist()
            num_frames = data.shape[0]

            feature = {
                'data': _float_feature(flattened_data),
                'label': _int64_feature(label),
                'num_frames': _int64_feature(num_frames)
            }

            example = tf.train.Example(features=tf.train.Features(feature=feature))
            writer.write(example.SerializeToString())
if not os.path.exists('tfrecords/train.tfrecord'):
    os.makedirs('tfrecords', exist_ok=True)

    print("Converting Train set to TFRecords...")
    create_tfrecord(train_paths, train_labels, 'tfrecords/train.tfrecord')
    
    print("Converting Val set to TFRecords...")
    create_tfrecord(val_paths, val_labels, 'tfrecords/val.tfrecord')
    
    print("Converting Test set to TFRecords...")
    create_tfrecord(test_paths, test_labels, 'tfrecords/test.tfrecord')
else:
    print("TFRecords already exist. Skipping conversion.")

TFRecords already exist. Skipping conversion.


Now we define a fast loader that uses `tf.data.TFRecordDataset`. Note that since sequences have variable lengths, we still use `padded_batch`.

In [17]:
def parse_tfrecord_fn(example):
    feature_description = {
        'data': tf.io.VarLenFeature(tf.float32),
        'label': tf.io.FixedLenFeature([], tf.int64),
        'num_frames': tf.io.FixedLenFeature([], tf.int64),
    }
    example = tf.io.parse_single_example(example, feature_description)

    # Convert sparse tensor to dense and reshape
    data = tf.sparse.to_dense(example['data'])
    num_frames = tf.cast(example['num_frames'], tf.int32)
    data = tf.reshape(data, [num_frames, 126])

    label = tf.one_hot(tf.cast(example['label'], tf.int32), depth=num_classes)
    return data, label

def load_tfrecord_dataset(file_path, batch_size=32, shuffle=True):
    ds = tf.data.TFRecordDataset(file_path)
    if shuffle:
        ds = ds.shuffle(1000)
    ds = ds.map(parse_tfrecord_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.padded_batch(batch_size, padded_shapes=([None, 126], [num_classes]))
    return ds.prefetch(tf.data.AUTOTUNE)

strategy = tf.distribute.MirroredStrategy()
print(f"Number of devices: {strategy.num_replicas_in_sync}")

BATCH_SIZE_PER_REPLICA = 64
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync

train_ds = load_tfrecord_dataset('tfrecords/train.tfrecord', batch_size=GLOBAL_BATCH_SIZE, shuffle=True)
val_ds = load_tfrecord_dataset('tfrecords/val.tfrecord', batch_size=GLOBAL_BATCH_SIZE, shuffle=False)
test_ds = load_tfrecord_dataset('tfrecords/test.tfrecord', batch_size=GLOBAL_BATCH_SIZE, shuffle=False)

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0',)
Number of devices: 1


In [6]:
print(num_classes)

502


In [9]:
data = np.load("/kaggle/input/datasets/adelhefny/arsl-landmarks-npy-part2/test/0001/01_01_0001_(10_11_16_16_20_30)_c.npy")
print(f"Number of frames: {data.shape[0]}")
print(f"Number of landmarks: {data.shape[1]}")

Number of frames: 8
Number of landmarks: 126


In [48]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model


class SinusoidalPositionalEncoding(layers.Layer):
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.supports_masking = True

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]

        pos = tf.cast(tf.range(seq_len)[:, tf.newaxis], tf.float32)  # (T, 1)
        i = tf.cast(tf.range(self.d_model)[tf.newaxis, :], tf.float32)  # (1, D)

        angle_rates = 1.0 / tf.pow(
            10000.0,
            (2.0 * tf.floor(i / 2.0)) / tf.cast(self.d_model, tf.float32)
        )
        angles = pos * angle_rates

        sines = tf.sin(angles[:, 0::2])
        cosines = tf.cos(angles[:, 1::2])

        # Interleave by concatenation, works for even and odd d_model
        pos_encoding = tf.concat([sines, cosines], axis=-1)
        pos_encoding = pos_encoding[:, :self.d_model]

        return inputs + pos_encoding[tf.newaxis, :, :]

    def compute_mask(self, inputs, mask=None):
        return mask


class NaNProtector(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.supports_masking = True

    def call(self, inputs):
        return tf.where(tf.math.is_finite(inputs), inputs, tf.zeros_like(inputs))

    def compute_mask(self, inputs, mask=None):
        return mask


class MaskedGlobalAveragePooling1D(layers.Layer):
    def call(self, inputs, mask=None):
        if mask is None:
            return tf.reduce_mean(inputs, axis=1)

        mask = tf.cast(mask, inputs.dtype)[:, :, tf.newaxis]
        summed = tf.reduce_sum(inputs * mask, axis=1)
        counts = tf.reduce_sum(mask, axis=1)
        return summed / tf.maximum(counts, 1.0)


class MaskedGlobalMaxPooling1D(layers.Layer):
    def call(self, inputs, mask=None):
        if mask is None:
            return tf.reduce_max(inputs, axis=1)

        mask = tf.cast(mask, inputs.dtype)[:, :, tf.newaxis]
        neg_inf = tf.cast(-1e9, inputs.dtype)
        masked_inputs = tf.where(mask > 0, inputs, neg_inf)
        pooled = tf.reduce_max(masked_inputs, axis=1)

        # If a sequence is fully masked, return zeros instead of -inf
        all_masked = tf.reduce_sum(mask, axis=1) == 0
        pooled = tf.where(all_masked, tf.zeros_like(pooled), pooled)
        return pooled


def build_temporal_transformer(num_landmarks, num_classes, d_model, num_heads, ff_dim, num_layers):
    inputs = layers.Input(shape=(None, num_landmarks))

    # Clean NaN / Inf at the gate
    x = layers.Lambda(
        lambda t: tf.where(tf.math.is_finite(t), t, tf.zeros_like(t))
    )(inputs)

    # Explicit padding mask from zero rows
    mask = layers.Lambda(
        lambda t: tf.reduce_any(tf.not_equal(t, 0.0), axis=-1)
    )(x)

    x = layers.Dense(d_model)(x)
    x = SinusoidalPositionalEncoding(d_model)(x)

    attn_mask = layers.Lambda(lambda m: m[:, tf.newaxis, :])(mask)

    for _ in range(num_layers):
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=0.1,
        )(x, x, attention_mask=attn_mask)

        attention_output = NaNProtector()(attention_output)
        x = layers.Add()([x, attention_output])
        x = layers.LayerNormalization(epsilon=1e-6)(x)

        ffn = layers.Dense(ff_dim, activation="relu")(x)
        ffn = layers.Dropout(0.1)(ffn)
        ffn = layers.Dense(d_model)(ffn)
        ffn = NaNProtector()(ffn)

        x = layers.Add()([x, ffn])
        x = layers.LayerNormalization(epsilon=1e-6)(x)

        # Keep padded positions quiet
        x = layers.Multiply()([
            x,
            layers.Lambda(lambda m: tf.cast(m[:, :, tf.newaxis], tf.float32))(mask)
        ])

    avg_pool = MaskedGlobalAveragePooling1D()(x, mask=mask)
    max_pool = MaskedGlobalMaxPooling1D()(x, mask=mask)

    x = layers.Concatenate()([avg_pool, max_pool])
    x = layers.Dense(d_model // 2, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return Model(inputs=inputs, outputs=outputs)

In [43]:
# --- 2. Configuration & Data Setup ---
NUM_LANDMARKS = 126
NUM_CLASSES = 502
WINDOW_SIZE = 30 # Simulating a 30-frame rolling buffer

model_configs = {
    "Base (486k)":   {"d_model": 128, "num_heads": 4, "ff_dim": 256,  "num_layers": 1},
    "Medium (3.5M)": {"d_model": 256, "num_heads": 8, "ff_dim": 1024, "num_layers": 4},
    "Large (20M)":   {"d_model": 512, "num_heads": 8, "ff_dim": 2048, "num_layers": 6}
}

# Create dummy data for the benchmark (Replace with your train_ds in production)
print("Generating dummy data for initialization...")
dummy_x = tf.random.normal((128, WINDOW_SIZE, NUM_LANDMARKS))
dummy_y = tf.one_hot(tf.random.uniform((128,), maxval=NUM_CLASSES, dtype=tf.int32), depth=NUM_CLASSES)
dummy_dataset = tf.data.Dataset.from_tensor_slices((dummy_x, dummy_y)).batch(32)

# --- 3. The Benchmark Loop ---
results = {}

for name, config in model_configs.items():
    print(f"\n{'='*50}\nBenchmarking: {name}\n{'='*50}")
    
    # 1. Build and Compile
    model = build_temporal_transformer(
        num_landmarks=NUM_LANDMARKS,
        num_classes=NUM_CLASSES,
        **config
    )
    
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    # 2. Train for 1 Epoch (Initializes graph and weights)
    print("Training 1 warmup epoch to initialize computational graph...")
    model.fit(dummy_dataset, epochs=1, verbose=0)
    
    # 3. Setup Inference Benchmark
    # Wrap in tf.function to simulate optimized execution graph
    @tf.function(jit_compile=True) # jit_compile=True forces XLA optimization
    def optimized_predict(input_tensor):
        return model(input_tensor, training=False)

    # Simulate a single sliding-window frame: shape (1, 30, 126)
    single_frame_sequence = tf.random.normal((1, WINDOW_SIZE, NUM_LANDMARKS))
    
    print("Warming up inference engine...")
    for _ in range(10):
        _ = optimized_predict(single_frame_sequence)
        
    print("Running latency test (1000 iterations)...")
    iterations = 1000
    
    # Wait for GPU to finish any lingering operations before starting the clock
    tf.test.experimental.sync_devices()
    start_time = time.perf_counter()
    
    for _ in range(iterations):
        _ = optimized_predict(single_frame_sequence)
        
    # Wait for all GPU operations to finish before stopping the clock
    tf.test.experimental.sync_devices()
    end_time = time.perf_counter()
    
    # 4. Calculate Results
    total_time = end_time - start_time
    avg_latency_ms = (total_time / iterations) * 1000
    
    results[name] = avg_latency_ms
    print(f"✅ Result for {name}: {avg_latency_ms:.2f} milliseconds per frame")

# --- 4. Final Report ---
print("\n" + "="*50)
print("FINAL INFERENCE LATENCY REPORT (Target: < 33ms)")
print("="*50)
for name, latency in results.items():
    fps_equivalent = 1000 / latency if latency > 0 else 0
    print(f"{name.ljust(15)} : {latency:>5.2f} ms | Max capability: {fps_equivalent:>5.0f} FPS")

Generating dummy data for initialization...

Benchmarking: Base (486k)


ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.ops`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```


In [49]:
from tensorflow import keras

NUM_LANDMARKS = 126
NUM_CLASSES = 502
WINDOW_SIZE = 30
with strategy.scope():
    landmark_transformer_model = build_temporal_transformer(
        num_landmarks=NUM_LANDMARKS,
        num_classes=NUM_CLASSES,
        **model_configs["Medium (3.5M)"]
    )

    landmark_transformer_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4,clipnorm=1.0),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=[keras.metrics.CategoricalAccuracy()]
    )

landmark_transformer_model.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_18      │ (None, None, 126) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, None, 126) │          0 │ input_layer_18[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_178 (Dense)   │ (None, None, 256) │     32,512 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, None)      │          0 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sinusoidal_positio… │ (None, None, 256) │          0 │ dense_178[0][0]   │
│ (SinusoidalPositio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 1, None)   │          0 │ lambda_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 256) │    263,168 │ sinusoidal_posit… │
│ (MultiHeadAttentio… │                   │            │ sinusoidal_posit… │
│                     │                   │            │ lambda_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ na_n_protector_48   │ (None, None, 256) │          0 │ multi_head_atten… │
│ (NaNProtector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_134 (Add)       │ (None, None, 256) │          0 │ sinusoidal_posit… │
│                     │                   │            │ na_n_protector_4… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 256) │        512 │ add_134[0][0]     │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_179 (Dense)   │ (None, None,      │    263,168 │ layer_normalizat… │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_148         │ (None, None,      │          0 │ dense_179[0][0]   │
│ (Dropout)           │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_180 (Dense)   │ (None, None, 256) │    262,400 │ dropout_148[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ na_n_protector_49   │ (None, None, 256) │          0 │ dense_180[0][0]   │
│ (NaNProtector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_135 (Add)       │ (None, None, 256) │          0 │ layer_normalizat… │
│                     │                   │            │ na_n_protector_4… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 256) │        512 │ add_135[0][0]     │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_3 (Lambda)   │ (None, None, 1)   │          0 │ lambda_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, None, 256) │          0 │ layer_normalizat… │
│                     │                   │            │ lambda_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 3,321,974 (12.67 MB)

 Trainable params: 3,321,974 (12.67 MB)

 Non-trainable params: 0 (0.00 B)

In [51]:
history = landmark_transformer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
    ]
)

Epoch 1/35
794/794 ━━━━━━━━━━━━━━━━━━━━ 35s 44ms/step - categorical_accuracy: 0.9735 - loss: 1.2577 - val_categorical_accuracy: 0.9586 - val_loss: 1.2172 - learning_rate: 1.0000e-04
Epoch 2/35
794/794 ━━━━━━━━━━━━━━━━━━━━ 35s 44ms/step - categorical_accuracy: 0.9730 - loss: 1.2560 - val_categorical_accuracy: 0.9601 - val_loss: 1.2036 - learning_rate: 1.0000e-04
Epoch 3/35
794/794 ━━━━━━━━━━━━━━━━━━━━ 35s 44ms/step - categorical_accuracy: 0.9735 - loss: 1.2493 - val_categorical_accuracy: 0.9646 - val_loss: 1.1914 - learning_rate: 1.0000e-04
Epoch 4/35
794/794 ━━━━━━━━━━━━━━━━━━━━ 35s 44ms/step - categorical_accuracy: 0.9751 - loss: 1.2425 - val_categorical_accuracy: 0.9606 - val_loss: 1.1988 - learning_rate: 1.0000e-04
Epoch 5/35
794/794 ━━━━━━━━━━━━━━━━━━━━ 35s 44ms/step - categorical_accuracy: 0.9771 - loss: 1.2388 - val_categorical_accuracy: 0.9657 - val_loss: 1.1927 - learning_rate: 1.0000e-04
Epoch 6/35
794/794 ━━━━━━━━━━━━━━━━━━━━ 35s 44ms/step - categorical_accuracy: 0.9779 - los

In [52]:
import numpy as np
import time

# Assuming you have the RealTimeSignPredictor from the previous step initialized
# predictor = RealTimeSignPredictor("model_quantized.tflite", window_size=30)

def simulate_realtime_inference(npy_file_path, predictor_instance, class_names_map):
    """
    Simulates a real-time video feed by passing rows of a .npy file 
    into the sliding window predictor one by one.
    """
    print(f"Loading pre-processed video landmarks from: {npy_file_path}")
    
    # 1. Load the data
    try:
        # Load the float16 array and convert to float32 for the model
        video_landmarks = np.load(npy_file_path).astype(np.float32)
    except Exception as e:
        print(f"Failed to load file: {e}")
        return

    total_frames = video_landmarks.shape[0]
    print(f"Successfully loaded {total_frames} frames (126 landmarks each).")
    
    # Clear any residual data in the buffer before starting a new video
    predictor_instance.reset()
    
    print("\nStarting simulated stream...")
    print("-" * 40)
    
    # 2. Simulate frame-by-frame processing
    for frame_idx, frame_data in enumerate(video_landmarks):
        prediction_id = predictor_instance.process_frame(frame_data)
        if prediction_id is not None:
            sign_word = class_names_map.get(prediction_id, f"Class {prediction_id}")
            print(f"Frame {frame_idx:03d} | Buffer Full | Prediction: {sign_word}")
        else:
            print(f"Frame {frame_idx:03d} | Filling Buffer ({len(predictor_instance.frames_buffer)}/{predictor_instance.window_size})...")

    print("-" * 40)
    print("Video stream complete.")

test_video_path = "/kaggle/input/datasets/adelhefny/arsl-landmarks-npy-part1/test/0002/01_02_0002_(15_11_16_16_42_51)_c.npy"

dummy_class_map = {i: f"Arabic_Sign_{i}" for i in range(502)}

simulate_realtime_inference(test_video_path, predictor, dummy_class_map)

NameError: name 'predictor' is not defined

In [54]:
import numpy as np
from sklearn.metrics import top_k_accuracy_score

print("Extracting raw predictions for deep analysis...")

y_true = []
y_pred_probs = []
for x_batch, y_batch in test_ds:
    predictions = landmark_transformer_model.predict(x_batch, verbose=0)
    batch_y_true = np.argmax(y_batch.numpy(), axis=1)
    y_true.extend(batch_y_true)
    y_pred_probs.extend(predictions)

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

top1_acc = np.mean(y_true == y_pred_classes)

all_labels = np.arange(NUM_CLASSES) 
top5_acc = top_k_accuracy_score(y_true, y_pred_probs, k=5, labels=all_labels)

print("\n" + "="*40)
print("FINAL TEST METRICS")
print("="*40)
print(f"Top-1 Accuracy : {top1_acc * 100:.2f}% (Exact Match)")
print(f"Top-5 Accuracy : {top5_acc * 100:.2f}% (In top 5 guesses)")
incorrect_mask = y_true != y_pred_classes
wrong_true_labels = y_true[incorrect_mask]
wrong_pred_labels = y_pred_classes[incorrect_mask]

print("\n" + "-"*40)
print("Error Analysis: Most Common Mistakes")
print("-" * 40)
mistake_pairs = list(zip(wrong_true_labels, wrong_pred_labels))
unique_mistakes, counts = np.unique(mistake_pairs, axis=0, return_counts=True)

sorted_indices = np.argsort(-counts)
for i in range(min(5, len(sorted_indices))):
    idx = sorted_indices[i]
    true_class = unique_mistakes[idx][0]
    pred_class = unique_mistakes[idx][1]
    count = counts[idx]
    print(f"True Sign [{true_class}] was misclassified as [{pred_class}] -> {count} times")

Extracting raw predictions for deep analysis...

FINAL TEST METRICS
Top-1 Accuracy : 97.83% (Exact Match)
Top-5 Accuracy : 99.22% (In top 5 guesses)

----------------------------------------
Error Analysis: Most Common Mistakes
----------------------------------------
True Sign [26] was misclassified as [27] -> 7 times
True Sign [27] was misclassified as [26] -> 5 times
True Sign [42] was misclassified as [104] -> 5 times
True Sign [13] was misclassified as [12] -> 3 times
True Sign [38] was misclassified as [104] -> 3 times


In [55]:
pip install tf2onnx onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 18.8 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [56]:
import tf2onnx
import onnx

WINDOW_SIZE = 30
NUM_LANDMARKS = 126

input_signature = [
    tf.TensorSpec([None, WINDOW_SIZE, NUM_LANDMARKS], tf.float32, name='input_landmarks')
]

onnx_model, _ = tf2onnx.convert.from_keras(landmark_transformer_model, input_signature, opset=15)
onnx_file_path = "arabic_sign_model_large.onnx"
onnx.save(onnx_model, onnx_file_path)

I0000 00:00:1778085177.644347      57 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1778085177.644591      57 single_machine.cc:374] Starting new session
I0000 00:00:1778085177.645904      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1778085178.415162      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1778085178.531216      57 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1778085178.531503      57 single_machine.cc:374] Starting new session
I0000 00:00:1778085178.532596      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 wi

In [ ]:
NUM_LANDMARKS = 126
NUM_CLASSES = 502
with strategy.scope():
    small_model = build_temporal_transformer(
        num_landmarks=NUM_LANDMARKS,
        num_classes=NUM_CLASSES,
        d_model=128, 
        num_heads=4, 
        ff_dim=256,  
        num_layers=1
    )
    
    small_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=[keras.metrics.CategoricalAccuracy()]
    )

small_model.summary()

In [20]:
from sklearn.metrics import top_k_accuracy_score
history = small_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', 
            patience=5, 
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', 
            factor=0.5, 
            patience=3
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath='best_small_model.keras',
            monitor='val_categorical_accuracy',
            save_best_only=True
        )
    ]
)

print("\n--- Running Final Evaluation ---")
test_loss, test_accuracy = small_model.evaluate(test_ds)
print(f"Base Test Loss: {test_loss:.4f}")

print("\nExtracting detailed predictions for Top-5 and Error Analysis...")
y_true = []
y_pred_probs = []

for x_batch, y_batch in test_ds:
    predictions = small_model.predict(x_batch, verbose=0)
    batch_y_true = np.argmax(y_batch.numpy(), axis=1)
    
    y_true.extend(batch_y_true)
    y_pred_probs.extend(predictions)

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

# Metrics
top1_acc = np.mean(y_true == y_pred_classes)
all_labels = np.arange(NUM_CLASSES)
top5_acc = top_k_accuracy_score(y_true, y_pred_probs, k=5, labels=all_labels)

print("\n========================================")
print("FINAL TEST METRICS (Base Model)")
print("========================================")
print(f"Top-1 Accuracy : {top1_acc * 100:.2f}%")
print(f"Top-5 Accuracy : {top5_acc * 100:.2f}%")

# Error Analysis
incorrect_mask = y_true != y_pred_classes
wrong_true_labels = y_true[incorrect_mask]
wrong_pred_labels = y_pred_classes[incorrect_mask]

print("\n----------------------------------------")
print("Error Analysis: Top 5 Confused Signs")
print("----------------------------------------")
if len(wrong_true_labels) > 0:
    mistake_pairs = list(zip(wrong_true_labels, wrong_pred_labels))
    unique_mistakes, counts = np.unique(mistake_pairs, axis=0, return_counts=True)
    sorted_indices = np.argsort(-counts)

    for i in range(min(5, len(sorted_indices))):
        idx = sorted_indices[i]
        true_class = unique_mistakes[idx][0]
        pred_class = unique_mistakes[idx][1]
        count = counts[idx]
        print(f"True Sign [{true_class}] was misclassified as [{pred_class}] -> {count} times")
else:
    print("Zero errors made on the test set!")

Epoch 1/25
397/397 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - categorical_accuracy: 0.8714 - loss: 1.8653 - val_categorical_accuracy: 0.9401 - val_loss: 1.5525 - learning_rate: 2.5000e-05
Epoch 2/25
397/397 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - categorical_accuracy: 0.8734 - loss: 1.8570 - val_categorical_accuracy: 0.9411 - val_loss: 1.5500 - learning_rate: 2.5000e-05
Epoch 3/25
397/397 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - categorical_accuracy: 0.8688 - loss: 1.8605 - val_categorical_accuracy: 0.9401 - val_loss: 1.5523 - learning_rate: 2.5000e-05
Epoch 4/25
397/397 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - categorical_accuracy: 0.8711 - loss: 1.8553 - val_categorical_accuracy: 0.9397 - val_loss: 1.5489 - learning_rate: 2.5000e-05
Epoch 5/25
397/397 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - categorical_accuracy: 0.8707 - loss: 1.8578 - val_categorical_accuracy: 0.9415 - val_loss: 1.5498 - learning_rate: 2.5000e-05
Epoch 6/25
397/397 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - categorical_accuracy: 0.8735 - los

In [19]:
import tf2onnx
import onnx

WINDOW_SIZE = 30
NUM_LANDMARKS = 126

input_signature = [
    tf.TensorSpec([None, WINDOW_SIZE, NUM_LANDMARKS], tf.float32, name='input_landmarks')
]

onnx_model, _ = tf2onnx.convert.from_keras(small_model, input_signature, opset=15)
onnx_file_path = "arabic_sign_model_small.onnx"
onnx.save(onnx_model, onnx_file_path)

I0000 00:00:1777625570.822355      57 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1777625570.822691      57 single_machine.cc:374] Starting new session
I0000 00:00:1777625570.837124      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777625570.838656      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1777625571.016853      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777625571.018398      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 M